# LAB 5 — Dashboard Naive RAG vs Advanced RAG
## Aula 5 — Avaliação de Pipelines RAG

Compara o **Naive RAG (#T01)** do LAB 2 com um **Advanced RAG** que adiciona **Hybrid Search (BM25 + kNN + RRF)** e **reranking via LLM** sobre o mesmo dataset de 50 pares.

**Stack:**
- **LLM gerador & judge**: Groq `llama-3.1-8b-instant` (fallback Ollama `llama3.2:3b`)
- **Embeddings**: Ollama `bge-m3` (1024d)
- **Vector store**: OpenSearch 3.x — índice TCU 2026
- **RRF pipeline**: criado em LAB2 da Aula 4 (`hybrid_rrf_pipeline`)

**Pré-requisito:** LAB 1 e LAB 2 executados.


## 1. Instalação + Stack


In [ ]:
import subprocess, sys
for pkg in ['ragas>=0.1.16','datasets','pandas','matplotlib','seaborn','tqdm',
            'langchain>=0.2','langchain-core>=0.2',
            'langchain-groq>=0.1','langchain-ollama>=0.1',
            'opensearch-py>=2.7','python-dotenv>=1.0','langfuse>=2.36']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('dependencias instaladas')


In [ ]:
import os, json, time, warnings, math
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for ep in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
            Path.cwd().parent.parent/'.env']:
    if ep.exists(): load_dotenv(ep, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY','')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL','llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL','http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL','llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL','bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST','localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT',9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER','admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD','admin')
INDEX_NAME         = os.getenv('INDEX_NAME','corpus_juridico_aula4')
PIPELINE_RRF       = os.getenv('PIPELINE_RRF','hybrid_rrf_pipeline')

META = {'faithfulness':0.80, 'answer_relevancy':0.75,
        'context_recall':0.70, 'context_precision':0.70}

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from opensearchpy import OpenSearch
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception: pass
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.0, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == 1024

os_client = OpenSearch(
    hosts=[{'host':OPENSEARCH_HOST,'port':OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER,OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
print(f'Stack OK | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL} | OS={os_client.info()["version"]["number"]} | index={INDEX_NAME}')


## 2. Carregar Baseline do LAB 2 (Naive RAG)


In [ ]:
LAB2_AGG = 'lab2_baseline_agregado.json'
LAB2_CSV = 'ragas_baseline_resultados.csv'
naive_agg = None
if os.path.exists(LAB2_AGG):
    with open(LAB2_AGG, encoding='utf-8') as f:
        naive_agg = json.load(f)
    print(f'naive baseline: {naive_agg}')
else:
    print(f'AVISO: {LAB2_AGG} nao encontrado - execute LAB2 antes')


## 3. Carregar Dataset de Avaliação


In [ ]:
DATASET = 'dataset_avaliacao_completo.json'
if not os.path.exists(DATASET):
    raise FileNotFoundError(f'{DATASET} nao encontrado. Execute LAB1.')
with open(DATASET, encoding='utf-8') as f:
    dataset = json.load(f)
N_PARES = 20  # subconjunto para o exemplo
amostra = dataset[:N_PARES]
print(f'{len(dataset)} pares totais - usando {len(amostra)} para Advanced RAG')


## 4. Pipeline Advanced RAG

Combina:
1. **Hybrid Search** com search pipeline RRF (`hybrid_rrf_pipeline` do LAB2 da Aula 4)
2. **Rerank via LLM**: o LLM ordena os top-k contextos por relevância
3. Geração com Groq/Ollama


In [ ]:
PROMPT_GEN = ChatPromptTemplate.from_messages([
    ('system','Voce e um assistente juridico brasileiro. Responda APENAS com base nos trechos.'),
    ('human','TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:')
])
GEN_CHAIN = PROMPT_GEN | llm | StrOutputParser()

PROMPT_RERANK = ChatPromptTemplate.from_messages([
    ('system','Voce reordena trechos juridicos por relevancia para a pergunta. '
              'Responda com numeros separados por virgula (mais relevante primeiro).'),
    ('human','PERGUNTA: {pergunta}\n\nTRECHOS:\n{lista}\n\n'
             'Devolva indices (1-based) ordenados, sem explicacao. Ex: 3,1,5,2,4')
])
RERANK_CHAIN = PROMPT_RERANK | llm | StrOutputParser()

def hybrid_search_rrf(q: str, k: int = 10):
    """Hybrid search com RRF pipeline (BM25 + kNN BGE-M3)."""
    v = embeddings.embed_query(q)
    body = {'size': k,
            'query':{'hybrid':{'queries':[
                {'match':{'conteudo':{'query':q}}},
                {'knn':{'embedding':{'vector':v, 'k':k}}}]}},
            '_source':['titulo','conteudo']}
    try:
        r = os_client.search(index=INDEX_NAME, body=body,
                              params={'search_pipeline': PIPELINE_RRF})
    except Exception:
        # Fallback: bool/should sem pipeline
        body2 = {'size': k, 'query': {'bool': {'should': [
                  {'match':{'conteudo':{'query':q}}},
                  {'knn':{'embedding':{'vector':v,'k':k}}}]}},
                 '_source':['titulo','conteudo']}
        r = os_client.search(index=INDEX_NAME, body=body2)
    return [(h['_source'].get('titulo','') + '\n' + h['_source'].get('conteudo','')).strip()
             for h in r['hits']['hits']]

def llm_rerank(q: str, ctxs: list, top_n: int = 5) -> list:
    if len(ctxs) <= top_n: return ctxs
    lista = '\n'.join(f'{i+1}. {c[:300]}' for i, c in enumerate(ctxs))
    try:
        order_str = RERANK_CHAIN.invoke({'pergunta':q,'lista':lista}).strip()
        idxs = []
        for tok in order_str.replace('\n',',').split(','):
            tok = tok.strip().split('.')[0].split(')')[0]
            if tok.isdigit(): idxs.append(int(tok)-1)
        idxs = [i for i in idxs if 0 <= i < len(ctxs)]
        seen = set(); final = []
        for i in idxs:
            if i not in seen: seen.add(i); final.append(ctxs[i])
        for i in range(len(ctxs)):
            if i not in seen: final.append(ctxs[i])
        return final[:top_n]
    except Exception:
        return ctxs[:top_n]

def advanced_rag(question: str, ground_truth: str, k: int = 10, top_n: int = 5) -> dict:
    ctxs_raw = hybrid_search_rrf(question, k=k)
    ctxs_top = llm_rerank(question, ctxs_raw, top_n=top_n)
    ctxs_top = ctxs_top or ['Contexto indisponivel.']
    bloco = '\n\n'.join(f'[Doc {i+1}]\n{c[:600]}' for i, c in enumerate(ctxs_top))
    try:
        ans = GEN_CHAIN.invoke({'contextos':bloco,'pergunta':question}).strip()
    except Exception as e:
        ans = f'[Erro geracao: {e}]'
    return {'question': question, 'answer': ans,
            'contexts': ctxs_top, 'ground_truth': ground_truth}

# smoke test
_t = advanced_rag(amostra[0]['question'], amostra[0]['ground_truth'])
print(f'smoke advanced OK | {len(_t["contexts"])} ctxs | answer: {_t["answer"][:120]}...')


## 5. Gerar respostas Advanced RAG para 20 pares + RAGAS


In [ ]:
from tqdm.auto import tqdm
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

advanced_records = []
t0 = time.time()
for par in tqdm(amostra, desc='Advanced RAG'):
    r = advanced_rag(par['question'], par['ground_truth'])
    r['id'] = par['id']; r['tipo'] = par['tipo']; r['dificuldade'] = par['dificuldade']
    advanced_records.append(r)
print(f'\n{len(advanced_records)} pares Advanced RAG gerados em {time.time()-t0:.0f}s')

# RAGAS para Advanced RAG
ds_adv = Dataset.from_list([{
    'question': r['question'], 'answer': r['answer'],
    'contexts': r['contexts'], 'ground_truth': r['ground_truth'],
} for r in advanced_records])

print('Calculando RAGAS para Advanced RAG...')
res_adv = evaluate(ds_adv,
                    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
                    llm=ragas_llm, embeddings=ragas_embeddings, raise_exceptions=False)
df_adv = res_adv.to_pandas()
for col in ['id','tipo','dificuldade']:
    df_adv[col] = [r[col] for r in advanced_records]

import pandas as pd
print('\n=== Médias Advanced RAG ===')
for m in META:
    print(f'  {m:22s} {df_adv[m].mean():.4f}')


## 6. Carregar mesmos 20 pares do Naive RAG (LAB2) para comparação justa


In [ ]:
df_naive_full = None
if os.path.exists(LAB2_CSV):
    df_naive_full = pd.read_csv(LAB2_CSV)
    if 'id' in df_naive_full.columns:
        ids_amostra = [r['id'] for r in advanced_records]
        df_naive = df_naive_full[df_naive_full['id'].isin(ids_amostra)].copy()
    else:
        df_naive = df_naive_full.head(len(advanced_records)).copy()
    print(f'naive: {len(df_naive)} linhas alinhadas')
else:
    print(f'AVISO: {LAB2_CSV} nao encontrado - usando agregado do lab2')
    if naive_agg:
        df_naive = pd.DataFrame([{m: naive_agg[m] for m in META} for _ in range(len(amostra))])
    else:
        raise RuntimeError('Sem baseline naive - execute LAB2.')


## 7. Dashboard Comparativo (Naive vs Advanced)


In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

medias_naive    = {m: df_naive[m].mean()    for m in META}
medias_advanced = {m: df_adv[m].mean()      for m in META}

print('=== Comparacao Naive vs Advanced ===')
print(f"{'metrica':<22}{'naive':>8}{'advanced':>10}{'delta':>9}{'meta':>8}")
for m, target in META.items():
    n = medias_naive[m]; a = medias_advanced[m]; d = a - n
    print(f'{m:<22}{n:>8.4f}{a:>10.4f}{d:>+9.4f}{target:>8.2f}')

# Grafico
metricas_lbl = ['Faithfulness','AnsRel','CtxRec','CtxPre']
naive_vals    = [medias_naive[m]    for m in META]
adv_vals      = [medias_advanced[m] for m in META]
metas         = [META[m] for m in META]
x = np.arange(len(metricas_lbl)); w = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - w/2, naive_vals, w, color='#3498db', alpha=0.85, label='Naive RAG (LAB2)')
ax.bar(x + w/2, adv_vals,   w, color='#2ecc71', alpha=0.85, label='Advanced RAG (LAB5)')
for i, t in enumerate(metas):
    ax.hlines(t, i-0.45, i+0.45, colors='red', linestyles='--', linewidths=2)
for i, (n, a) in enumerate(zip(naive_vals, adv_vals)):
    ax.text(i-w/2, n+0.01, f'{n:.3f}', ha='center', fontsize=9)
    ax.text(i+w/2, a+0.01, f'{a:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metricas_lbl)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score (0-1)')
ax.set_title(f'Naive vs Advanced RAG ({LLM_PROVIDER}/{LLM_MODEL})', fontweight='bold')
ax.grid(axis='y', alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('dashboard_naive_vs_advanced.png', dpi=140); plt.show()
print('dashboard_naive_vs_advanced.png salvo')


## 8. Heatmap por Tipo Jurídico


In [ ]:
if 'tipo' in df_naive.columns and 'tipo' in df_adv.columns:
    pivot_naive = df_naive.groupby('tipo')[list(META)].mean()
    pivot_adv   = df_adv.groupby('tipo')[list(META)].mean()
    delta = pivot_adv - pivot_naive
    print('\n=== Delta (Advanced - Naive) por tipo ===')
    print(delta.round(4))
    fig, ax = plt.subplots(figsize=(10, max(4, len(delta)*0.5)))
    sns.heatmap(delta, annot=True, fmt='+.3f', cmap='RdYlGn', center=0,
                cbar_kws={'label':'delta'}, linewidths=.5, ax=ax)
    ax.set_title('Δ por tipo jurídico (Advanced - Naive)', fontweight='bold')
    plt.tight_layout(); plt.savefig('lab5_heatmap_delta.png', dpi=140); plt.show()


## 9. Tabela-Resumo + Status por Meta


In [ ]:
resumo = []
for m, target in META.items():
    n = medias_naive[m]; a = medias_advanced[m]
    resumo.append({
        'metrica': m, 'meta': target,
        'naive': round(n, 4), 'advanced': round(a, 4),
        'delta': round(a - n, 4),
        'naive_status':    'OK' if n >= target else 'ABAIXO',
        'advanced_status': 'OK' if a >= target else 'ABAIXO',
    })
df_resumo = pd.DataFrame(resumo)
print(df_resumo.to_string(index=False))
df_resumo.to_csv('lab5_resumo_naive_vs_advanced.csv', index=False, encoding='utf-8')
df_adv.to_csv('lab5_advanced_resultados.csv', index=False, encoding='utf-8')
print('\nlab5_resumo_naive_vs_advanced.csv + lab5_advanced_resultados.csv salvos')


## 10. Conclusões

Em corpus jurídico denso (TCU 2026):

- **Hybrid Search + RRF + Rerank** tende a melhorar **Context Precision** e **Faithfulness** vs Naive
- O **rerank via LLM** custa ~1 chamada adicional/consulta — vale quando Faithfulness < 0.80
- A melhoria **varia por tipo jurídico**: terminologia técnica (administrativo, processual) ganha mais

## Referências

ES, S. et al. **RAGAS**. arXiv:2309.15217, 2023.  
BOGAN, R. et al. **Introducing RRF for hybrid search**. OpenSearch Blog, 2025.  
GROQ INC. **Groq API**. <https://console.groq.com/docs>.
